# 01 — Click Interactions

The last lesson introduced the event system and showed how to extract coordinates from a click. This lesson uses those coordinates to **build things** — markers with labels, growing lists of points, a live path that draws itself as you click, and finally a GeoJSON export of everything you collected.

```
click  →  lat/lon  →  create feature  →  add to map
```

## Setup

In [25]:
from ipyleaflet import Map, Marker, Polyline, GeoJSON
from ipywidgets import HTML

WICHITA_FALLS = (33.9137, -98.4934)

## Capturing Click Coordinates

Quick recap of the pattern from the last lesson — filter to `"click"` events, unpack `coordinates`.

```
click  →  kwargs["coordinates"]  →  (lat, lon)
```

Everything in this lesson starts from this two-line extract:

In [26]:
def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    lat, lon = kwargs["coordinates"]   # always this line
    print(f"lat={lat:.5f}  lon={lon:.5f}")

m = Map(center=WICHITA_FALLS, zoom=12)
m.on_interaction(on_click)
m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Numbered Markers with Popups

A bare marker is hard to tell apart once you have placed several. Add a sequential number and a popup that shows the exact coordinates.

In [3]:
m = Map(center=WICHITA_FALLS, zoom=12)
count = [0]  # list so the closure can mutate it

def place_numbered_marker(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    count[0] += 1

    marker = Marker(location=(lat, lon), title=f"Point {count[0]}")
    marker.popup = HTML(value=f"""
        <b>Point {count[0]}</b><br>
        lat: {lat:.5f}<br>
        lon: {lon:.5f}
    """)
    m.add(marker)

m.on_interaction(place_numbered_marker)
m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

Click several locations, then click each marker to see its popup.

## Storing Clicked Points

A marker is a visual object on the map — but it lives in ipyleaflet, not in Python. To use your clicked coordinates later (for computation, export, or building geometry), collect them into a plain Python list.

```python
clicked_points = []   # grows with every click
```

Each click appends one `(lat, lon)` tuple.

In [4]:
m = Map(center=WICHITA_FALLS, zoom=12)
clicked_points = []

def collect_point(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    clicked_points.append((lat, lon))

    # Visual confirmation on the map
    m.add(Marker(location=(lat, lon), title=f"Point {len(clicked_points)}"))

    # Text feedback below the cell
    print(f"[{len(clicked_points)}]  ({lat:.5f}, {lon:.5f})")

m.on_interaction(collect_point)
m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

In [5]:
# Inspect the list after clicking
print(f"{len(clicked_points)} points collected:")
for i, (lat, lon) in enumerate(clicked_points):
    print(f"  {i+1}.  ({lat:.5f}, {lon:.5f})")

0 points collected:


## Building a Live Path

Once coordinates are stored in a list, you can connect them into a line. A `Polyline` takes a list of `(lat, lon)` tuples and draws them as a connected path.

The trick: update the `Polyline`'s `locations` property each time a new point arrives. Because the polyline is a widget, the map redraws instantly — no cell re-run.

In [32]:
m = Map(center=WICHITA_FALLS, zoom=12)

path_points = []
path_line = Polyline(locations=[], color="#e74c3c", weight=3)
m.add(path_line)

def build_path(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    path_points.append((lat, lon))

    # Drop a marker at the new point
    m.add(Marker(location=(lat, lon), title=f"Point {len(path_points)}"))

    # Update the line — widget redraws automatically
    path_line.locations = path_points.copy()

    print(f"Path has {len(path_points)} point(s)")

m.on_interaction(build_path)
m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

Click several locations in sequence. Each click extends the red line to the new point.

The list of points in `path_points` is the data behind what you see — the map is just visualizing it.

## Exporting Clicked Points as GeoJSON

Collected coordinates are only useful if you can use them elsewhere. Convert `path_points` to a GeoJSON FeatureCollection and save it to a file.

Remember: GeoJSON coordinates are `[lon, lat]` — the reverse of ipyleaflet's `(lat, lon)`.

In [33]:
import json

def points_to_geojson(points):
    """Convert a list of (lat, lon) tuples to a GeoJSON FeatureCollection."""
    features = [
        {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [lon, lat]   # GeoJSON order: [lon, lat]
            },
            "properties": {"index": i + 1}
        }
        for i, (lat, lon) in enumerate(points)
    ]
    return {"type": "FeatureCollection", "features": features}

# Convert and inspect
fc = points_to_geojson(path_points)
print(json.dumps(fc, indent=2))

{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "geometry": {
        "type": "Point",
        "coordinates": [
          -98.48144531250001,
          33.925414577686475
        ]
      },
      "properties": {
        "index": 1
      }
    },
    {
      "type": "Feature",
      "geometry": {
        "type": "Point",
        "coordinates": [
          -98.42994689941408,
          33.91943194771954
        ]
      },
      "properties": {
        "index": 2
      }
    },
    {
      "type": "Feature",
      "geometry": {
        "type": "Point",
        "coordinates": [
          -98.47732543945314,
          33.88894250077001
        ]
      },
      "properties": {
        "index": 3
      }
    }
  ]
}


In [34]:
# Save to file
output_path = "clicked_points.geojson"

with open(output_path, "w") as f:
    json.dump(fc, f, indent=2)

print(f"Saved {len(path_points)} points to {output_path}")

Saved 3 points to clicked_points.geojson


In [35]:
# Reload the saved file and display it — closes the loop
with open(output_path) as f:
    saved = json.load(f)

m2 = Map(center=WICHITA_FALLS, zoom=12)
m2.add(GeoJSON(data=saved))
m2

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Exercise A

Click the `build_path` map above to collect at least 3 points. Then, in the cell below, convert `path_points` into a single **`LineString`** GeoJSON feature (not a FeatureCollection of Points). Print the result.

Remember: GeoJSON coordinate order is `[lon, lat]`.

In [36]:
import json

m2 = Map(center=WICHITA_FALLS, zoom=12)
path_points = []
path_line = Polyline(locations=[], color="#e74c3c", weight=3)
m2.add(path_line)



def build_path(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    path_points.append((lon, lat))

    # Drop a marker at the new point
    m.add(Marker(location=(lat, lon), title=f"Point {len(path_points)}"))

    # Update the line — widget redraws automatically
    path_line.locations = path_points.copy()

    print(f"Path has {len(path_points)} point(s)")

m2.on_interaction(build_path)
m2

coordinates = [[lon, lat] for lat, lon in path_points]
linestring_feature = {
    "type": "Feature",
    "geometry": {
        "type": "LineString",
        "coordinates": coordinates
    },
    "properties": {}
}

print(json.dumps(linestring_feature, indent=2))
# After clicking the map above to collect path_points, run this cell.
# Convert path_points into a single LineString GeoJSON feature (not a collection of Points).
# Print the result.
# Your code here

{
  "type": "Feature",
  "geometry": {
    "type": "LineString",
    "coordinates": []
  },
  "properties": {}
}


## Exercise B

Add an **Undo** button to the live path builder. Each click of the button should:

1. Remove the last marker from the map
2. Pop the last entry from the points list
3. Update the `Polyline` to reflect the shorter path

In [29]:
from ipyleaflet import Map, Marker, Polyline
from ipywidgets import Button, HBox, VBox

m2 = Map(center=WICHITA_FALLS, zoom=12)
path_points = []
path_line = Polyline(locations=[], color="#e74c3c", weight=3)
m2.add(path_line)

markers = []  # Track markers so we can remove them on undo

def build_path(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    path_points.append((lat, lon))

    # Drop a marker and keep a reference to it
    marker = Marker(location=(lat, lon), title=f"Point {len(path_points)}")
    m2.add(marker)
    markers.append(marker)

    # Update the line
    path_line.locations = path_points.copy()

    print(f"Path has {len(path_points)} point(s)")

m2.on_interaction(build_path)

# Undo button
undo_btn = Button(description="Undo", button_style="warning", icon="undo")

def on_undo(b):
    if not path_points:
        print("Nothing to undo.")
        return

    # Remove last point and its marker
    path_points.pop()
    last_marker = markers.pop()
    m2.remove(last_marker)

    # Update the line
    path_line.locations = path_points.copy()

    print(f"Undid last point. Path now has {len(path_points)} point(s).")

undo_btn.on_click(on_undo)

display(VBox([undo_btn, m2]))

---

## Check Your Understanding

Build a map where:

1. Each click drops a numbered marker and appends `(lat, lon)` to a list
2. After collecting at least three points, run a second cell that prints each coordinate and exports them to `my_points.geojson`

```python
# your answer here

```

In [37]:
from ipyleaflet import Map, Marker
from ipywidgets import HTML
import json

m3 = Map(center=WICHITA_FALLS, zoom=12)
clicked_points = []

def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    
    lat, lon = kwargs["coordinates"]
    clicked_points.append((lat, lon))
    
    # Numbered marker with a popup label
    label = HTML(value=f"<b>{len(clicked_points)}</b>")
    marker = Marker(location=(lat, lon), title=f"Point {len(clicked_points)}", popup=label)
    m3.add(marker)
    
    print(f"Point {len(clicked_points)}: ({lat:.5f}, {lon:.5f})")

m3.on_interaction(on_click)
m3

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

In [38]:
assert len(clicked_points) >= 3, f"Need at least 3 points, only have {len(clicked_points)}"

# Print each coordinate
print(f"Collected {len(clicked_points)} points:\n")
for i, (lat, lon) in enumerate(clicked_points, start=1):
    print(f"  {i}. lat={lat:.5f}, lon={lon:.5f}")

# Build GeoJSON FeatureCollection of Points
geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [lon, lat]  # GeoJSON order: [lon, lat]
            },
            "properties": {"id": i}
        }
        for i, (lat, lon) in enumerate(clicked_points, start=1)
    ]
}

output_path = "my_points.geojson"
with open(output_path, "w") as f:
    json.dump(geojson, f, indent=2)

print(f"\nExported to {output_path}")

Collected 3 points:

  1. lat=33.92342, lon=-98.54290
  2. lat=33.92912, lon=-98.46668
  3. lat=33.89151, lon=-98.49930

Exported to my_points.geojson


## Next

In [02 — Dynamic Layers](./02-Dynamic_Layers.ipynb), we add and remove entire GeoJSON layers at runtime — toggling, swapping, and clearing them while the map is live.